## Academic portfolio context

**Student / portfolio maintainer:** Mohammed Alaa Eddine GUERBOUZ  
**Institution:** INGETIS (Paris, France)  
**Programme:** Master-level studies · Generative AI / Large Language Models  
**Academic year:** 2026–2027  
**Topic:** Structured Outputs and LLM Evaluation  
**Material type:** Course practical notebook / learning exercise  

**Problem explored:** How can an application reliably consume an LLM answer and check response quality?

> Portfolio annotation added in October 2026. Original instructional content and attribution are preserved below. This header identifies the student maintaining this study portfolio; it is **not** a claim that all classroom template code was originally authored by the student. The source filename was `jour_1_evaluation_sortie_structuree_colab_fr.ipynb`. See the folder README for requirements and limitations.


# Jour 1 - Évaluation et sortie structurée

## Présentation

Bienvenue dans ce nouveau notebook du cours sur l'IA générative. Vous allez découvrir quelques techniques pour évaluer les réponses d'un modèle de langage. Pour cette évaluation, vous utiliserez aussi la capacité de Gemini à produire des **données structurées**, afin d'obtenir les résultats sous forme d'objets Python.

Remarque : ce notebook contient plus de code que le premier notebook du jour 1 (Prompting). Il n'est pas indispensable pour la suite du cours : vous pouvez le passer ou y revenir plus tard dans la semaine. Si vous n'avez pas encore fait le notebook Prompting, commencez par celui-là : il présente les bases de l'interaction avec les LLM.

Consultez aussi le **livre blanc bonus** [Evaluating Large Language Models](https://services.google.com/fh/files/blogs/neurips_evaluation.pdf) (en anglais).

> **Version adaptée pour Google Colab** : ce notebook est en français, lit la clé API depuis les **secrets Colab**, relance automatiquement les appels en cas de quota atteint et utilise le modèle `gemini-3.1-flash-lite` (les modèles Gemini 2.0 d'origine ont été arrêtés). Les prompts d'évaluation restent en anglais : ce sont des modèles de prompts officiels de Google Cloud.

## Configuration

Installez le SDK Python.

In [ ]:
# Colab inclut déjà google-genai : on le met simplement à jour vers la dernière version
# (nécessaire pour les modèles Gemini récents).
!pip install -U -q "google-genai"

In [ ]:
from google import genai
from google.genai import types

from IPython.display import Markdown, display

genai.__version__

# Modèle utilisé dans tout le notebook.
MODEL = "gemini-3.1-flash-lite"

### Configurer votre clé API

Pour exécuter la cellule suivante, votre clé API doit être enregistrée dans un [secret Colab](https://colab.research.google.com/notebooks/secrets.ipynb) nommé `GOOGLE_API_KEY` :

1. Dans la barre latérale gauche de Colab, cliquez sur l'icône **clé** 🔑 (« Secrets »).
2. Cliquez sur **Ajouter un secret**, saisissez `GOOGLE_API_KEY` comme nom et collez votre clé comme valeur.
3. Activez l'interrupteur **Accès au notebook**.

Si vous n'avez pas encore de clé API, vous pouvez en obtenir une sur [AI Studio](https://aistudio.google.com/app/apikey). Vous trouverez des [instructions détaillées dans la documentation](https://ai.google.dev/gemini-api/docs/api-key).

Hors de Colab (exécution locale, VM…), la cellule utilise la variable d'environnement `GOOGLE_API_KEY`.

In [ ]:
import os

GOOGLE_API_KEY = None

# 1) Secrets Colab (icône clé dans la barre latérale gauche)
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # On n'est pas dans Colab

# 2) Variable d'environnement (exécution locale, VM, etc.)
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "Clé API introuvable : ajoutez un secret Colab nommé GOOGLE_API_KEY "
        "et activez « Accès au notebook »."
    )

client = genai.Client(api_key=GOOGLE_API_KEY)
print("Clé API chargée ✅")

Si vous obtenez une erreur du type `SecretNotFoundError` ou `NotebookAccessError`, vérifiez que le secret s'appelle exactement `GOOGLE_API_KEY` **et** que l'interrupteur **Accès au notebook** est activé dans le panneau Secrets (icône clé). Colab peut aussi afficher une fenêtre vous demandant d'autoriser l'accès : cliquez sur **Autoriser l'accès**.

### Nouvelle tentative automatique (retry)

Ce notebook envoie beaucoup de requêtes : mettez en place une nouvelle tentative automatique quand le quota par minute est atteint. En cas d'erreur 429 (quota) ou 503 (modèle surchargé), l'appel est relancé automatiquement après 2 s, 4 s, 8 s… pendant 60 s au maximum. Il n'y a aucune attente quand l'appel réussit.

> **Clé gratuite** : le tier gratuit est partagé et peut être saturé pendant le cours. Si une cellule met un peu de temps à répondre, c'est normal : laissez le retry faire son travail. Si l'erreur persiste après 60 s, relancez simplement la cellule.

In [ ]:
from google.api_core import retry


is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

# Politique commune : 2 s, 4 s, 8 s... (15 s max entre deux essais), 60 s au total.
politique_retry = retry.Retry(predicate=is_retriable, initial=2, maximum=15, timeout=60)

# On garde la fonction d'origine pour pouvoir réexécuter cette cellule sans empiler les retry.
if not hasattr(genai.models.Models, "_generate_original"):
    genai.models.Models._generate_original = genai.models.Models.generate_content

genai.models.Models.generate_content = politique_retry(genai.models.Models._generate_original)

## Évaluation

Quand on utilise des LLM dans des cas réels, il est important de savoir s'ils fonctionnent bien. Comme les LLM génèrent du texte libre, beaucoup de cas sont difficiles à mesurer. Dans ce notebook, vous allez parcourir quelques techniques simples pour évaluer les réponses d'un LLM et comprendre ses performances.

Dans cet exemple, vous allez évaluer une tâche de résumé à partir du [rapport technique de Gemini 1.5 Pro](https://storage.googleapis.com/cloud-samples-data/generative-ai/pdf/2403.05530.pdf). Commencez par télécharger le PDF dans l'environnement du notebook, puis envoyez cette copie à l'API Gemini.

In [ ]:
!wget -nv -O gemini.pdf https://storage.googleapis.com/cloud-samples-data/generative-ai/pdf/2403.05530.pdf

document_file = client.files.upload(file='gemini.pdf')

### Résumer un document

La demande de résumé utilisée ici est assez basique : elle cible le processus d'entraînement, sans autre consigne.

In [ ]:
request = 'Tell me about the training process used here.'

def summarise_doc(request: str) -> str:
  """Exécute la demande sur le document envoyé."""
  # Température basse pour stabiliser la réponse.
  config = types.GenerateContentConfig(temperature=0.0)
  response = client.models.generate_content(
      model=MODEL,
      config=config,
      contents=[request, document_file],
  )

  return response.text

summary = summarise_doc(request)
Markdown(summary)

### Définir un évaluateur

Pour une tâche comme celle-ci, vous pouvez évaluer plusieurs aspects : la façon dont le modèle a suivi le prompt (« respect des instructions »), le fait qu'il s'appuie sur les données fournies (« ancrage », *groundedness*), la facilité de lecture du texte (« fluidité »), ou d'autres critères comme la « verbosité » ou la « qualité ».

Vous pouvez demander à un LLM de réaliser cette évaluation comme vous le feriez avec un évaluateur humain : avec une définition claire et une [grille d'évaluation](https://fr.wikipedia.org/wiki/Grille_d%27%C3%A9valuation).

Dans cette étape, vous définissez un agent d'évaluation à partir d'un prompt de « résumé » déjà rédigé, et vous l'utilisez pour mesurer la qualité du résumé généré.

Remarque : pour d'autres prompts d'évaluation prêts à l'emploi (ancrage, sécurité, cohérence, etc.), consultez cette [liste complète de prompts d'évaluation par modèle](https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates) dans la documentation Google Cloud.

In [ ]:
import enum

# Prompt d'évaluation (laissé en anglais : modèle officiel de Google Cloud)
SUMMARY_PROMPT = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by AI models.
We will provide you with the user input and an AI-generated responses.
You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will assign the response a rating following the Rating Rubric and Evaluation Steps. Give step-by-step explanations for your rating, and only choose ratings from the Rating Rubric.

# Evaluation
## Metric Definition
You will be assessing summarization quality, which measures the overall ability to summarize text. Pay special attention to length constraints, such as in X words or in Y sentences. The instruction for performing a summarization task and the context to be summarized are provided in the user prompt. The response should be shorter than the text in the context. The response should not contain information that is not present in the context.

## Criteria
Instruction following: The response demonstrates a clear understanding of the summarization task instructions, satisfying all of the instruction's requirements.
Groundedness: The response contains information included only in the context. The response does not reference any outside information.
Conciseness: The response summarizes the relevant details in the original text without a significant loss in key information without being too verbose or terse.
Fluency: The response is well-organized and easy to read.

## Rating Rubric
5: (Very good). The summary follows instructions, is grounded, is concise, and fluent.
4: (Good). The summary follows instructions, is grounded, concise, and fluent.
3: (Ok). The summary mostly follows instructions, is grounded, but is not very concise and is not fluent.
2: (Bad). The summary is grounded, but does not follow the instructions.
1: (Very bad). The summary is not grounded.

## Evaluation Steps
STEP 1: Assess the response in aspects of instruction following, groundedness, conciseness, and verbosity according to the criteria.
STEP 2: Score based on the rubric.

# User Inputs and AI-generated Response
## User Inputs

### Prompt
{prompt}

## AI-generated Response
{response}
"""

# Énumération structurée pour récupérer le résultat.
class SummaryRating(enum.Enum):
  VERY_GOOD = '5'
  GOOD = '4'
  OK = '3'
  BAD = '2'
  VERY_BAD = '1'


def eval_summary(prompt, ai_response):
  """Évalue le résumé généré par rapport au prompt utilisé."""

  chat = client.chats.create(model=MODEL)

  # Génère la réponse textuelle complète.
  response = chat.send_message(
      message=SUMMARY_PROMPT.format(prompt=prompt, response=ai_response)
  )
  verbose_eval = response.text

  # Convertit la réponse dans la structure voulue.
  structured_output_config = types.GenerateContentConfig(
      response_mime_type="text/x.enum",
      response_schema=SummaryRating,
  )
  response = chat.send_message(
      message="Convert the final score.",
      config=structured_output_config,
  )
  structured_eval = response.parsed

  return verbose_eval, structured_eval


text_eval, struct_eval = eval_summary(prompt=[request, document_file], ai_response=summary)
Markdown(text_eval)

Dans cet exemple, le modèle a produit une justification textuelle dans le cadre d'une conversation (chat). Cette réponse complète est utile à la fois pour la lecture humaine et pour donner au modèle un espace où « prendre des notes » pendant qu'il évalue le texte et produit la note finale. Cette stratégie de « prise de notes » ou de « réflexion » fonctionne généralement bien avec les modèles autorégressifs, où le texte généré est réinjecté dans le modèle à chaque étape de génération : les « notes » sont donc utilisées pour produire le résultat final.

Au tour suivant, le modèle convertit sa réponse textuelle en réponse structurée. Si vous voulez agréger des notes ou les utiliser dans du code, mieux vaut éviter d'analyser du texte libre. Ici, on passe le schéma `SummaryRating` : le modèle convertit l'historique de la conversation en une instance de l'énumération `SummaryRating`.

In [1]:
struct_eval

NameError: name 'struct_eval' is not defined

### Améliorer ou dégrader le prompt de résumé

Les modèles Gemini sont en général assez bons pour résumer directement, sans beaucoup de consignes : attendez-vous à un résultat `GOOD` ou `VERY_GOOD` sur la tâche précédente, même avec un prompt rudimentaire. Exécutez-la plusieurs fois pour vous faire une idée de la réponse moyenne.

Pour voir comment influencer le résumé, réfléchissez à ce que vous pourriez changer dans la demande. Le prompt d'évaluation `SUMMARY_PROMPT` peut vous donner des idées.

Essayez les modifications suivantes et observez leur effet, positif ou négatif, sur le résultat :
* Précisez la longueur du résumé,
* Demandez des informations précises,
* Demandez des informations absentes du document,
* Demandez différents niveaux de résumé (par exemple « explique comme à un enfant de 5 ans » ou « avec tous les détails techniques »).

In [ ]:
new_prompt = "Explain like I'm 5 the training process"
# Essayez :
#  ELI5 the training process
#  Summarise the needle/haystack evaluation technique in 1 line
#  Describe the model architecture to someone with a civil engineering degree
#  What is the best LLM?

if not new_prompt:
  raise ValueError("Définissez un nouveau prompt de résumé.")


def run_and_eval_summary(prompt):
  """Génère puis évalue le résumé avec le nouveau prompt."""
  summary = summarise_doc(new_prompt)
  display(Markdown(summary + '\n-----'))

  text, struct = eval_summary([new_prompt, document_file], summary)
  display(Markdown(text + '\n-----'))
  print(struct)

run_and_eval_summary(new_prompt)

## L'évaluation en pratique

L'évaluation a de nombreux usages concrets, par exemple :
* Itérer rapidement sur un prompt avec un petit ensemble de documents de test,
* Comparer différents modèles pour trouver celui qui convient le mieux, par exemple le meilleur compromis prix/performance, ou la meilleure performance sur une tâche précise,
* Vérifier qu'un changement de modèle ou de prompt dans un système en production ne dégrade pas la qualité (non-régression).

Dans cette section, vous allez essayer deux approches d'évaluation différentes.

### Évaluation ponctuelle (pointwise)

La technique utilisée plus haut, qui consiste à évaluer une seule paire entrée/sortie selon des critères, s'appelle l'évaluation ponctuelle (*pointwise*). Elle sert à juger une réponse de façon absolue : « est-ce bon ou mauvais ? ».

Dans cet exercice, vous allez tester différents prompts de consigne sur un ensemble de questions.

In [ ]:
import functools

# Essayez ces consignes, ou modifiez-les et ajoutez les vôtres.
terse_guidance = "Answer the following question in a single sentence, or as close to that as possible."
moderate_guidance = "Provide a brief answer to the following question, use a citation if necessary, but only enough to answer the question."
cited_guidance = "Provide a thorough, detailed answer to the following question, citing the document and supplying additional background information as much as possible."
guidance_options = {
    'Terse': terse_guidance,
    'Moderate': moderate_guidance,
    'Cited': cited_guidance,
}

questions = [
    # Décommentez une ou plusieurs questions, ou ajoutez les vôtres.
    # Évaluer plus de questions prend plus de temps, mais donne des résultats
    # plus fiables. En production, on peut avoir des centaines de questions
    # pour évaluer un système complexe.

    # "What metric(s) are used to evaluate long context performance?",
    "How does the model perform on code tasks?",
    "How many layers does it have?",
    # "Why is it called Gemini?",
]

if not questions:
  raise NotImplementedError('Ajoutez des questions à évaluer.')


@functools.cache
def answer_question(question: str, guidance: str = '') -> str:
  """Répond à la question à partir du document envoyé et de la consigne."""
  config = types.GenerateContentConfig(
      temperature=0.0,
      system_instruction=guidance,
  )
  response = client.models.generate_content(
      model=MODEL,
      config=config,
      contents=[question, document_file],
  )

  return response.text


answer = answer_question(questions[0], terse_guidance)
Markdown(answer)

Mettez maintenant en place un évaluateur de questions-réponses, comme précédemment, mais avec le [prompt d'évaluation ponctuelle QA](https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates#pointwise_question_answering_quality).

In [ ]:
import enum

QA_PROMPT = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by AI models.
We will provide you with the user prompt and an AI-generated responses.
You should first read the user prompt carefully for analyzing the task, and then evaluate the quality of the responses based on and rules provided in the Evaluation section below.

# Evaluation
## Metric Definition
You will be assessing question answering quality, which measures the overall quality of the answer to the question in the user prompt. Pay special attention to length constraints, such as in X words or in Y sentences. The instruction for performing a question-answering task is provided in the user prompt. The response should not contain information that is not present in the context (if it is provided).

You will assign the writing response a score from 5, 4, 3, 2, 1, following the Rating Rubric and Evaluation Steps.
Give step-by-step explanations for your scoring, and only choose scores from 5, 4, 3, 2, 1.

## Criteria Definition
Instruction following: The response demonstrates a clear understanding of the question answering task instructions, satisfying all of the instruction's requirements.
Groundedness: The response contains information included only in the context if the context is present in the user prompt. The response does not reference any outside information.
Completeness: The response completely answers the question with sufficient detail.
Fluent: The response is well-organized and easy to read.

## Rating Rubric
5: (Very good). The answer follows instructions, is grounded, complete, and fluent.
4: (Good). The answer follows instructions, is grounded, complete, but is not very fluent.
3: (Ok). The answer mostly follows instructions, is grounded, answers the question partially and is not very fluent.
2: (Bad). The answer does not follow the instructions very well, is incomplete or not fully grounded.
1: (Very bad). The answer does not follow the instructions, is wrong and not grounded.

## Evaluation Steps
STEP 1: Assess the response in aspects of instruction following, groundedness,completeness, and fluency according to the criteria.
STEP 2: Score based on the rubric.

# User Inputs and AI-generated Response
## User Inputs
### Prompt
{prompt}

## AI-generated Response
{response}
"""

class AnswerRating(enum.Enum):
  VERY_GOOD = '5'
  GOOD = '4'
  OK = '3'
  BAD = '2'
  VERY_BAD = '1'


@functools.cache
def eval_answer(prompt, ai_response, n=1):
  """Évalue la réponse générée par rapport à la question posée."""
  chat = client.chats.create(model=MODEL)

  # Génère la réponse textuelle complète.
  response = chat.send_message(
      message=QA_PROMPT.format(prompt=[prompt, document_file], response=ai_response)
  )
  verbose_eval = response.text

  # Convertit la réponse dans la structure voulue.
  structured_output_config = types.GenerateContentConfig(
      response_mime_type="text/x.enum",
      response_schema=AnswerRating,
  )
  response = chat.send_message(
      message="Convert the final score.",
      config=structured_output_config,
  )
  structured_eval = response.parsed

  return verbose_eval, structured_eval


text_eval, struct_eval = eval_answer(prompt=questions[0], ai_response=answer)
display(Markdown(text_eval))
print(struct_eval)

Exécutez maintenant l'évaluation en boucle. Notez que la consigne est cachée à l'agent d'évaluation. Si vous la lui transmettiez, il noterait le respect de cette consigne. Or, ici, l'objectif est de trouver le meilleur résultat global par rapport à la question de l'utilisateur, et non par rapport à l'instruction du développeur.

In [ ]:
import collections
import itertools

# Nombre de répétitions de chaque tâche, pour réduire l'erreur et calculer une moyenne.
# L'augmenter prend plus de temps mais donne de meilleurs résultats : essayez 2 ou 3.
NUM_ITERATIONS = 1

scores = collections.defaultdict(int)
responses = collections.defaultdict(list)

for question in questions:
  display(Markdown(f'## {question}'))
  for guidance, guide_prompt in guidance_options.items():

    for n in range(NUM_ITERATIONS):
      # Génère une réponse.
      answer = answer_question(question, guide_prompt)

      # Évalue la réponse (la consigne n'est pas transmise).
      written_eval, struct_eval = eval_answer(question, answer, n)
      print(f'{guidance}: {struct_eval}')

      # Enregistre la note numérique.
      scores[guidance] += int(struct_eval.value)

      # Enregistre les réponses, si vous voulez les consulter.
      responses[(guidance, question)].append((answer, written_eval))


Agrégez maintenant les notes pour voir les performances de chaque prompt.

In [ ]:
for guidance, score in scores.items():
  avg_score = score / (NUM_ITERATIONS * len(questions))
  nearest = AnswerRating(str(round(avg_score)))
  print(f'{guidance}: {avg_score:.2f} - {nearest.name}')

### Évaluation par paires (pairwise)

Le prompt d'évaluation ponctuelle utilisé à l'étape précédente comporte 5 niveaux de note. Cela peut être trop grossier pour votre système, ou vous voulez peut-être améliorer un prompt déjà « très bon ».

Une autre approche consiste à comparer deux réponses entre elles. C'est l'évaluation par paires (*pairwise*), une étape clé des algorithmes de classement et de tri : elle permet de classer vos prompts, à la place de l'approche ponctuelle ou en complément.

Cette étape met en œuvre l'évaluation par paires avec le [prompt de qualité QA par paires](https://cloud.google.com/vertex-ai/generative-ai/docs/models/metrics-templates#pairwise_question_answering_quality) de la documentation Google Cloud.

In [ ]:
QA_PAIRWISE_PROMPT = """\
# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by two AI models. We will provide you with the user input and a pair of AI-generated responses (Response A and Response B). You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.

You will first judge responses individually, following the Rating Rubric and Evaluation Steps. Then you will give step-by-step explanations for your judgment, compare results to declare the winner based on the Rating Rubric and Evaluation Steps.

# Evaluation
## Metric Definition
You will be assessing question answering quality, which measures the overall quality of the answer to the question in the user prompt. Pay special attention to length constraints, such as in X words or in Y sentences. The instruction for performing a question-answering task is provided in the user prompt. The response should not contain information that is not present in the context (if it is provided).

## Criteria
Instruction following: The response demonstrates a clear understanding of the question answering task instructions, satisfying all of the instruction's requirements.
Groundedness: The response contains information included only in the context if the context is present in the user prompt. The response does not reference any outside information.
Completeness: The response completely answers the question with sufficient detail.
Fluent: The response is well-organized and easy to read.

## Rating Rubric
"A": Response A answers the given question as per the criteria better than response B.
"SAME": Response A and B answers the given question equally well as per the criteria.
"B": Response B answers the given question as per the criteria better than response A.

## Evaluation Steps
STEP 1: Analyze Response A based on the question answering quality criteria: Determine how well Response A fulfills the user requirements, is grounded in the context, is complete and fluent, and provides assessment according to the criterion.
STEP 2: Analyze Response B based on the question answering quality criteria: Determine how well Response B fulfills the user requirements, is grounded in the context, is complete and fluent, and provides assessment according to the criterion.
STEP 3: Compare the overall performance of Response A and Response B based on your analyses and assessment.
STEP 4: Output your preference of "A", "SAME" or "B" to the pairwise_choice field according to the Rating Rubric.
STEP 5: Output your assessment reasoning in the explanation field.

# User Inputs and AI-generated Responses
## User Inputs
### Prompt
{prompt}

# AI-generated Response

### Response A
{baseline_model_response}

### Response B
{response}
"""


class AnswerComparison(enum.Enum):
  A = 'A'
  SAME = 'SAME'
  B = 'B'


@functools.cache
def eval_pairwise(prompt, response_a, response_b, n=1):
  """Détermine la meilleure de deux réponses à la même question."""

  chat = client.chats.create(model=MODEL)

  # Génère la réponse textuelle complète.
  response = chat.send_message(
      message=QA_PAIRWISE_PROMPT.format(
          prompt=[prompt, document_file],
          baseline_model_response=response_a,
          response=response_b)
  )
  verbose_eval = response.text

  # Convertit la réponse dans la structure voulue.
  structured_output_config = types.GenerateContentConfig(
      response_mime_type="text/x.enum",
      response_schema=AnswerComparison,
  )
  response = chat.send_message(
      message="Convert the final score.",
      config=structured_output_config,
  )
  structured_eval = response.parsed

  return verbose_eval, structured_eval


question = questions[0]
answer_a = answer_question(question, terse_guidance)
answer_b = answer_question(question, cited_guidance)

text_eval, struct_eval = eval_pairwise(
    prompt=question,
    response_a=answer_a,
    response_b=answer_b,
)

display(Markdown(text_eval))
print(struct_eval)

Une fois l'évaluateur par paires en place, il suffit d'un comparateur pour classer les prompts entre eux.

Cet exemple implémente les comparateurs minimaux nécessaires à un ordre total (`==` et `<`) et effectue la comparaison avec `n_iterations` évaluations sur l'ensemble des `questions`.

In [ ]:
@functools.total_ordering
class QAGuidancePrompt:
  """Un prompt de consigne (ou instruction système) pour les questions-réponses."""

  def __init__(self, prompt, questions, n_comparisons=NUM_ITERATIONS):
    """Crée le prompt. Fournit les questions d'évaluation et le nombre d'évaluations à effectuer."""
    self.prompt = prompt
    self.questions = questions
    self.n = n_comparisons

  def __str__(self):
    return self.prompt

  def _compare_all(self, other):
    """Compare deux prompts sur toutes les questions, sur n essais."""
    results = [self._compare_n(other, q) for q in questions]
    mean = sum(results) / len(results)
    return round(mean)

  def _compare_n(self, other, question):
    """Compare deux prompts sur une question, sur n essais."""
    results = [self._compare(other, question, n) for n in range(self.n)]
    mean = sum(results) / len(results)
    return mean

  def _compare(self, other, question, n=1):
    """Compare deux prompts sur une seule question."""
    answer_a = answer_question(question, self.prompt)
    answer_b = answer_question(question, other.prompt)

    _, result = eval_pairwise(
        prompt=question,
        response_a=answer_a,
        response_b=answer_b,
        n=n,  # Contourne le cache
    )
    # print(f'q[{question}], a[{self.prompt[:20]}...], b[{other.prompt[:20]}...]: {result}')

    # Convertit l'énumération en valeurs de comparaison numériques standard de Python.
    if result is AnswerComparison.A:
      return 1
    elif result is AnswerComparison.B:
      return -1
    else:
      return 0

  def __eq__(self, other):
    """Test d'égalité par évaluation par paires."""
    if not isinstance(other, QAGuidancePrompt):
      return NotImplemented

    return self._compare_all(other) == 0

  def __lt__(self, other):
    """Test d'ordre par évaluation par paires."""
    if not isinstance(other, QAGuidancePrompt):
      return NotImplemented

    return self._compare_all(other) < 0


Les fonctions de tri de Python fonctionnent maintenant directement sur les instances de `QAGuidancePrompt`. Les fonctions `answer_question` et `eval_pairwise` sont [mémoïsées](https://fr.wikipedia.org/wiki/M%C3%A9mo%C3%AFsation) pour éviter de régénérer inutilement les mêmes réponses ou évaluations : cette étape devrait donc être rapide, sauf si vous avez modifié les questions, les prompts ou le nombre d'itérations.

In [ ]:
terse_prompt = QAGuidancePrompt(terse_guidance, questions)
moderate_prompt = QAGuidancePrompt(moderate_guidance, questions)
cited_prompt = QAGuidancePrompt(cited_guidance, questions)

# Tri en ordre inverse, pour que le meilleur soit en premier
sorted_results = sorted([terse_prompt, moderate_prompt, cited_prompt], reverse=True)
for i, p in enumerate(sorted_results):
  if i:
    print('---')

  print(f'#{i+1}: {p}')

## Difficultés

### Limites des LLM

Les LLM ont des difficultés connues sur certaines tâches, et ces difficultés persistent quand on les utilise comme évaluateurs. Par exemple, les LLM ont du mal à compter les caractères d'un mot (c'est un problème numérique, pas un problème de langue) : un évaluateur LLM ne pourra donc pas évaluer correctement ce type de tâche. Il existe parfois des solutions, comme brancher des outils pour les problèmes mal adaptés à un modèle de langage. Mais il est important de connaître ces limites et de faire appel à des évaluateurs humains pour calibrer votre système d'évaluation et établir une référence.

Les évaluateurs LLM fonctionnent bien notamment parce que toutes les informations nécessaires sont dans le contexte d'entrée : le modèle n'a qu'à s'appuyer sur ces informations pour produire le résultat. Quand vous personnalisez des prompts d'évaluation ou construisez vos propres systèmes, gardez cela en tête : ne comptez pas sur les « connaissances internes » du modèle, ni sur un comportement qu'un outil fournirait mieux.

### Gagner en confiance

Pour rendre vos évaluations plus fiables, vous pouvez utiliser des évaluateurs variés : les mêmes prompts et les mêmes réponses, mais évalués par différents modèles, comme Gemini Flash et Pro, voire par différents fournisseurs, comme Gemini, Claude, ChatGPT ou des modèles locaux comme Gemma ou Qwen. C'est la même idée que la répétition des essais pour recueillir plusieurs « avis » et [réduire l'erreur](https://fr.wikipedia.org/wiki/Loi_des_grands_nombres), mais avec des modèles différents, les « avis » sont plus variés.

